In [0]:
# 01_ingest_data

from pyspark.sql import functions as F

catalog = "workspace"
schema = "stroke_prediction"

source_table = f"{catalog}.{schema}.healthcare_dataset_stroke_data"
raw_table = f"{catalog}.{schema}.stroke_raw"

required_columns = {
    "id",
    "gender",
    "age",
    "hypertension",
    "heart_disease",
    "ever_married",
    "work_type",
    "Residence_type",
    "avg_glucose_level",
    "bmi",
    "smoking_status",
    "stroke"
}

stroke_raw = spark.table(source_table)

missing_columns = required_columns - set(stroke_raw.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if stroke_raw.filter(F.col("id").isNull()).limit(1).count() > 0:
    raise ValueError("Missing patient IDs detected.")

if (
    stroke_raw
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError("Duplicated patient IDs detected.")

if (
    stroke_raw
    .filter(F.col("stroke").isNull() | ~F.col("stroke").isin(0, 1))
    .limit(1)
    .count() > 0
):
    raise ValueError("Invalid stroke values detected.")

if (
    stroke_raw
    .filter(F.col("age").isNotNull() & ~F.col("age").between(0, 120))
    .limit(1)
    .count() > 0
):
    raise ValueError("Invalid age values detected.")

(
    stroke_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(raw_table)
)

display(stroke_raw.limit(10))

print(f"Rows: {stroke_raw.count()}")
print(f"Columns: {len(stroke_raw.columns)}")
print(f"Saved as: {raw_table}")